<a href="https://colab.research.google.com/github/nicoavilan/Clases/blob/main/Redes_Neuronales_Demanda_Energia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a las redes neuronales para modelar datos


## Caso de estudio: demanda diaria de energía del Sistema Interconectado Nacional (SIN), 2021–2024

**Herramientas:** Python, `pandas`, `matplotlib`, `scikit-learn`, `TensorFlow/Keras`, `holidays`


> **Pregunta guía:** ¿podemos pronosticar la demanda de energía de mañana en Colombia con una red neuronal, y qué necesitamos hacer para que funcione mejor que un modelo sencillo?

## 0. Configuración del entorno

Este notebook funciona en Google Colab o en una instalación local de Jupyter. En Colab, TensorFlow ya viene instalado; solo hace falta instalar `holidays` (calendario de festivos de Colombia) y subir el archivo `Demanda_Energia_SIN_2021_2024.xlsx` al panel de archivos.

**Herramientas que usaremos:**

- **pandas**: lectura y manipulación de tablas (el archivo de Excel).
- **matplotlib**: gráficas.
- **holidays**: festivos oficiales de Colombia (incluye los trasladados a lunes por la Ley Emiliani).
- **scikit-learn**: escalado de variables, regresión lineal y métricas de error.
- **TensorFlow / Keras**: construcción y entrenamiento de redes neuronales. Keras es la interfaz de alto nivel de TensorFlow: describimos la red capa por capa y la biblioteca se encarga de los gradientes.

Importo librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow import keras

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)   # fija semillas de Python, NumPy y TensorFlow
print("TensorFlow", tf.__version__)

## 1. Exploración de los datos

El archivo contiene, para cada día entre el 1 de enero de 2021 y el 31 de diciembre de 2024:

- **Demanda de energía del SIN** (kWh): nuestra **variable objetivo**.
- **Precio de bolsa nacional** ($/kWh): promedio diario del precio en el mercado mayorista.

La hoja tiene un título y una fila vacía antes de los encabezados, por eso usamos `header=3`. Convertiremos la demanda a **GWh** (1 GWh = 10⁶ kWh) para trabajar con números más legibles.

In [ ]:
RUTA = "https://raw.githubusercontent.com/nicoavilan/Bases-de-datos/main/Demanda_Energia_SIN_2021_2024.xlsx"

datos = pd.read_excel(RUTA, header=3, usecols=[0, 1, 3])
datos.columns = ["fecha", "demanda_kWh", "precio"]
datos["fecha"] = pd.to_datetime(datos["fecha"])
datos = datos.set_index("fecha")
datos["demanda"] = datos["demanda_kWh"] / 1e6 # GWh

print(datos.shape)
print("Valores faltantes:\n", datos.isna().sum())
datos.head()

In [ ]:
datos[["demanda", "precio"]].describe().round(2)

### 1.1 La serie de tiempo

Antes de modelar, **siempre** se mira la serie. Fíjense en tres cosas: la tendencia de largo plazo, la "banda" que forma la variación semanal y los valles pronunciados.

In [ ]:
fig, ax = plt.subplots()
ax.plot(datos.index, datos["demanda"], lw=0.7, label="Diaria")
ax.plot(datos.index, datos["demanda"].rolling(30, center=True).mean(), lw=2,
                color="C3", label="Media móvil 30 días")
ax.set_ylabel("Demanda (GWh/día)")
ax.set_title("Demanda diaria de energía del SIN")
ax.legend();

In [ ]:
print("Demanda media por año (GWh/día):")
print(datos["demanda"].groupby(datos.index.year).mean().round(1))

En en código anterior puede también evaluar valores máximos, mínimos o dispersiones usando: `.max()`, `.min()`, `.std()`, `.var()`.

**Observación 1 – Tendencia:** la demanda media crece año tras año (de ≈ 203 a ≈ 224 GWh/día). Esto será **muy importante** más adelante: los datos de 2024 tienen niveles que el modelo nunca vio en 2021–2022.



### 1.2 Patrón semanal y festivos

In [ ]:
festivos_co = holidays.Colombia(years=range(2021, 2025))
datos["festivo"] = [int(d in festivos_co) for d in datos.index]
datos["dia_semana"] = datos.index.dayofweek          # 0 = lunes, ..., 6 = domingo

nombres = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
laborables = datos[datos["festivo"] == 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
laborables.boxplot(column="demanda", by="dia_semana", ax=axes[0])
axes[0].set_xticklabels(nombres); axes[0].set_title("Días no festivos, por día de la semana")
axes[0].set_xlabel(""); axes[0].set_ylabel("GWh/día")

datos.boxplot(column="demanda", by="festivo", ax=axes[1])
axes[1].set_xticklabels(["No festivo", "Festivo"]); axes[1].set_title("Festivos vs. no festivos")
axes[1].set_xlabel("")
plt.suptitle("");
print(f"Número de festivos en el periodo: {datos['festivo'].sum()}")

**Observación 2 – Calendario:** los domingos y festivos tienen una demanda claramente menor; el sábado es intermedio. Un festivo en lunes se parece más a un domingo que a un lunes: hay **interacciones** entre variables, algo que las redes neuronales capturan bien.

### 1.3 ¿Y el precio de bolsa?

In [ ]:
plt.plot(datos.index, datos["precio"], lw=1.7, color="C2")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(datos.index, datos["precio"], lw=0.7, color="C2")
axes[0].set_title("Precio de bolsa nacional ($/kWh)")
axes[1].scatter(datos["precio"], datos["demanda"], s=5, alpha=0.4, c=datos.index.year)
axes[1].set_xlabel("Precio ($/kWh)"); axes[1].set_ylabel("Demanda (GWh/día)")
axes[1].set_title("Demanda vs. precio (color = año)");
print("Correlación demanda–precio:", round(datos[["demanda", "precio"]].corr().iloc[0, 1], 3))

In [ ]:
plt.scatter(datos["precio"], datos["demanda"], s=5, alpha=0.4, c=datos.index.year)
plt.xlabel("Precio ($/kWh)")
plt.ylabel("Demanda (GWh/día)")
plt.title("Demanda vs. precio (color = año)");
print("Correlación demanda–precio:", round(datos[["demanda", "precio"]].corr().iloc[0, 1],3))

**Observación 3 – Precio:** la correlación positiva (≈ 0,44) se debe sobre todo a que ambas variables subieron en el tiempo, no a una relación causal directa día a día. Además, el precio de un día se forma *al mismo tiempo* que su demanda: usarlo para pronosticar la demanda de ese mismo día sería **fuga de información** (*data leakage*).

> 💬 **Pregunta para discusión:** si quisiéramos pronosticar la demanda de *mañana*, ¿qué información tendríamos disponible *hoy*?

## 2. Conceptos: ¿qué es una red neuronal?

### 2.1 La neurona artificial

Una neurona recibe un vector de entradas $\mathbf{x} = (x_1, \dots, x_n)$ y calcula

$$
z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}^\top \mathbf{x} + b,
\qquad a = \sigma(z).
$$

- $\mathbf{w}$ son los **pesos** y $b$ el **sesgo** (*bias*): son los **parámetros** que se aprenden.
- $\sigma$ es la **función de activación**: introduce no linealidad.

$a$ es la salida de la neurona. En una capa oculta, $a$ alimenta a la siguiente capa; solo la salida de la última capa es la predicción $\hat y$.

Si $\sigma$ es la identidad, una neurona es exactamente una **regresión lineal**. Toda la potencia de las redes viene de combinar muchas neuronas con activaciones no lineales.



### 2.2 Funciones de activación

In [ ]:
z = np.linspace(-4, 4, 200)
activaciones = {
    "ReLU: max(0, z)": np.maximum(0, z),
    "Sigmoide: 1/(1+e^{-z})": 1 / (1 + np.exp(-z)),
    "Tangente hiperbólica": np.tanh(z),
}
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (nombre, valor) in zip(axes, activaciones.items()):
    ax.plot(z, valor, lw=2); ax.set_title(nombre); ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
plt.tight_layout();

Hoy usaremos **ReLU** en las capas ocultas: es simple, rápida y es el estándar actual. En la capa de salida, como queremos un número real cualquiera (regresión), **no** usamos activación (activación lineal).

### 2.3 Perceptrón multicapa (MLP)

Una red *densa* organiza las neuronas en capas. Cada neurona de una capa recibe **todas** las salidas de la capa anterior:

$$
\mathbf{h}^{(1)} = \text{ReLU}\big(W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}\big), \quad
\mathbf{h}^{(2)} = \text{ReLU}\big(W^{(2)}\mathbf{h}^{(1)} + \mathbf{b}^{(2)}\big), \quad
\hat{y} = W^{(3)}\mathbf{h}^{(2)} + b^{(3)}.
$$

```
 Entradas (n)         Capa oculta 1 (32)     Capa oculta 2 (16)     Salida (1)
   x1  ─┐                 ○                      ○
   x2  ─┼──────────────▶  ○  ──────────────────▶ ○  ─────────────────▶  ŷ
   ...  │   (todas con     ...   (todas con       ...
   xn  ─┘    todas)       ○      todas)          ○
```

Una red es, en el fondo, una **función paramétrica** $\hat{y} = f(\mathbf{x};\theta)$ construida componiendo transformaciones afines y no linealidades. El **teorema de aproximación universal** garantiza que, con suficientes neuronas, un MLP puede aproximar cualquier función continua en un dominio acotado. Retengan la palabra *acotado*: volverá en la sección 5.



### 2.4 ¿Cómo aprende? Pérdida y descenso de gradiente

1. **Función de pérdida.** Medimos qué tan mal predice la red. Para regresión usamos el error cuadrático medio:
$$
L(\theta) = \frac{1}{N}\sum_{i=1}^{N}\big(y_i - \hat{y}_i\big)^2 .
$$
2. **Descenso de gradiente.** Movemos los parámetros en la dirección que más reduce la pérdida:
$$
\theta \leftarrow \theta - \eta\, \nabla_\theta L(\theta),
$$
donde $\eta$ es la **tasa de aprendizaje** (*learning rate*).
3. **Retropropagación (*backpropagation*).** Es la regla de la cadena aplicada capa por capa, desde la salida hacia la entrada, para calcular $\nabla_\theta L$ de forma eficiente. Keras lo hace automáticamente.

**Vocabulario de entrenamiento:**
- **Lote (*batch*)**: subconjunto de ejemplos con el que se calcula cada actualización del gradiente.
- **Época (*epoch*)**: una pasada completa por todos los datos de entrenamiento.
- **Optimizador**: la variante del descenso de gradiente. Usaremos **Adam**, que adapta la tasa de aprendizaje para cada parámetro.



### 2.5 Demostración: una neurona entrenada "a mano" con NumPy

Antes de usar Keras, entrenemos **una sola neurona lineal** que predice la demanda de hoy a partir de la demanda de hace 7 días. Ambas variables están estandarizadas (media 0, desviación 1). Aquí escribimos nosotros mismos los gradientes.

In [ ]:
# Datos para la demostración: demanda de hace 7 días -> demanda de hoy (solo 2021-2022)
demo = datos.loc[:"2022", ["demanda"]].copy()
demo["hace_7_dias"] = demo["demanda"].shift(7)
demo = demo.dropna() # Elimina valores nulos

Imrpimo el encabezado de los datos para verificar

In [ ]:
demo.head(10).round(2)

In [ ]:
# Estandarización
x = ((demo["hace_7_dias"] - demo["hace_7_dias"].mean()) / demo["hace_7_dias"].std()).values
y = ((demo["demanda"] - demo["demanda"].mean()) / demo["demanda"].std()).values

w, b = 0.0, 0.0          # parámetros iniciales
eta = 0.05               # tasa de aprendizaje
historia = []
for paso in range(100):
    y_hat = w * x + b                      # 1) predicción (forward pass)
    error = y_hat - y
    perdida = np.mean(error**2)            # 2) pérdida MSE
    dL_dw = 2 * np.mean(error * x)         # 3) gradientes (derivadas parciales)
    dL_db = 2 * np.mean(error)
    w -= eta * dL_dw                       # 4) actualización
    b -= eta * dL_db
    historia.append(perdida)

print(f"w = {w:.3f}, b = {b:.3f}, pérdida final = {historia[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historia); axes[0].set_xlabel("Paso"); axes[0].set_ylabel("MSE"); axes[0].set_title("La pérdida disminuye")
axes[1].scatter(x, y, s=6, alpha=0.4); xs = np.linspace(x.min(), x.max(), 2)
axes[1].plot(xs, w * xs + b, color="C3", lw=2); axes[1].set_xlabel("Demanda hace 7 días (estandarizada)")
axes[1].set_ylabel("Demanda hoy (estandarizada)"); axes[1].set_title("Recta aprendida");

> 💬 **Para experimentar en clase:** cambien `eta` a 0.5, 1.0 y 1.1. ¿Qué pasa con la pérdida? Una tasa demasiado grande hace que el algoritmo "salte" sobre el mínimo y diverja.

Una red neuronal hace exactamente esto, pero con miles de parámetros, activaciones no lineales y gradientes calculados por retropropagación.

## 3. Preparación de los datos

### 3.1 Formulación del problema

**Tarea:** pronosticar la demanda del día $t$ usando únicamente información conocida al final del día $t-1$ (pronóstico a un día).

### 3.2 Ingeniería de variables

Las redes trabajan con números, así que transformamos el calendario en variables numéricas:

| Variable | Tipo | Por qué |
|---|---|---|
| `dia_0` … `dia_6` | *One-hot* del día de la semana | El día de la semana es categórico: el domingo (6) no es "6 veces" el lunes (0). |
| `festivo` | Binaria | Los festivos tienen demanda baja. |
| `festivo_hace_7` | Binaria | Indica si el mismo día de la semana anterior fue festivo (afecta al rezago de 7 días). |
| `sin_anio`, `cos_anio` | Codificación cíclica | El 31 de diciembre y el 1 de enero son vecinos; seno y coseno capturan esa circularidad. |
| `lag1`, `lag7` | Rezagos | Demanda de ayer y de hace una semana. |
| `nivel` | Media móvil | Promedio de los 7 días anteriores: resume el "nivel" reciente de la demanda. |

Todas usan solo información del pasado (o del calendario, que se conoce de antemano).

In [ ]:
df = datos[["demanda", "festivo", "dia_semana"]].copy()

# Calendario
for k in range(7):
    df[f"dia_{k}"] = (df["dia_semana"] == k).astype(int)

# Codificación cíclica
dia_anio = df.index.dayofyear
df["sin_anio"] = np.sin(2 * np.pi * dia_anio / 365.25)
df["cos_anio"] = np.cos(2 * np.pi * dia_anio / 365.25)
df["festivo_hace_7"] = df["festivo"].shift(7)

# Información del pasado (shift garantiza que no usamos el día t)
df["lag1"] = df["demanda"].shift(1)
df["lag7"] = df["demanda"].shift(7)
df["nivel"] = df["demanda"].shift(1).rolling(7).mean()

df = df.dropna()
df.head(10).round(3)

### 3.3 Partición temporal: entrenamiento, validación y prueba

Con series de tiempo **no** se parten los datos al azar: eso permitiría al modelo "ver el futuro". Usamos bloques cronológicos:

- **Entrenamiento (2021–2022):** se ajustan los parámetros.
- **Validación (2023):** se decide cuándo detener el entrenamiento y se comparan configuraciones.
- **Prueba (2024):** se usa **una sola vez** al final, como si fuera el futuro real.

In [ ]:
entren = df.loc[:"2022"].copy()
valid  = df.loc["2023"].copy()
prueba = df.loc["2024"].copy()
print(f"Entrenamiento: {len(entren)} días | Validación: {len(valid)} días | Prueba: {len(prueba)} días")

fig, ax = plt.subplots()
for parte, nombre, c in [(entren, "Entrenamiento", "C0"), (valid, "Validación", "C1"), (prueba, "Prueba", "C3")]:
    ax.plot(parte.index, parte["demanda"], lw=0.7, color=c, label=nombre)
ax.set_ylabel("GWh/día"); ax.legend(); ax.set_title("Partición cronológica de los datos");

### 3.4 Escalado

Las redes entrenan mucho mejor cuando todas las variables tienen escalas comparables. Usamos `StandardScaler`, que transforma cada variable en $(x-\mu)/s$.

⚠️ **Regla de oro:** la media $\mu$ y la desviación $s$ se calculan **solo con el conjunto de entrenamiento** y luego se aplican a validación y prueba. Calcularlas con todos los datos es otra forma de fuga de información.

### 3.5 Métricas de error

- **MAE** (error absoluto medio), en GWh/día: $\frac{1}{N}\sum |y_i-\hat y_i|$.
- **MAPE** (error porcentual absoluto medio): $\frac{100\%}{N}\sum \frac{|y_i-\hat y_i|}{y_i}$. Fácil de comunicar: "nos equivocamos en promedio un X %".

In [ ]:
VARIABLES_CAL = ["festivo"] + [f"dia_{k}" for k in range(7)] + ["sin_anio", "cos_anio", "festivo_hace_7"]
VARIABLES_1 = VARIABLES_CAL + ["lag1", "lag7", "nivel"]
OBJETIVO_1 = "demanda"

esc_X1 = StandardScaler().fit(entren[VARIABLES_1])
esc_y1 = StandardScaler().fit(entren[[OBJETIVO_1]])

X_ent1, X_val1, X_pru1 = (esc_X1.transform(p[VARIABLES_1]) for p in (entren, valid, prueba))
y_ent1, y_val1 = (esc_y1.transform(p[[OBJETIVO_1]]) for p in (entren, valid))

resultados = {}   # aquí guardaremos el desempeño de cada modelo en 2024

def evaluar(nombre, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    mape = 100 * mean_absolute_percentage_error(y_real, y_pred)
    resultados[nombre] = {"MAE (GWh/día)": round(mae, 2), "MAPE (%)": round(mape, 2)}
    print(f"{nombre:40s}  MAE = {mae:5.2f} GWh/día   MAPE = {mape:4.2f} %")

print(f"{len(VARIABLES_1)} variables de entrada:", VARIABLES_1)

## 4. Modelos de referencia

Un modelo complejo solo vale la pena si supera a uno simple. Usaremos dos referencias:

1. **Ingenuo estacional:** "mañana será igual que el mismo día de la semana pasada" ($\hat y_t = y_{t-7}$).
2. **Regresión lineal** con las mismas variables que usará la red (equivale a una red sin capas ocultas).

In [ ]:
y_prueba = prueba["demanda"].values

evaluar("Ingenuo estacional (y_{t-7})", y_prueba, prueba["lag7"].values)

lineal_1 = LinearRegression().fit(X_ent1, y_ent1)
pred_lineal_1 = esc_y1.inverse_transform(lineal_1.predict(X_pru1)).ravel()
evaluar("Regresión lineal (objetivo directo)", y_prueba, pred_lineal_1)

## 5. Primera red neuronal con Keras

### 5.1 Construcción de la arquitectura

Keras ofrece `Sequential`, donde la red se describe como una lista de capas. La red 1 tiene:

- `keras.Input(shape=(14,))`: la capa de entrada, con 14 variables.
- `layers.Dense(32, activation="relu")`: primera capa oculta, con 32 neuronas ReLU.
- `layers.Dense(16, activation="relu")`: segunda capa oculta, con 16 neuronas ReLU.
- `layers.Dense(1)`: neurona de salida sin activación (regresión).

En el código, las capas ocultas se crean con un ciclo sobre `capas=(32, 16)`, de modo que cambiar la arquitectura es tan simple como cambiar esa tupla.

In [ ]:
def construir_red(n_entradas, capas=(32, 16), tasa=1e-3):
    modelo = keras.Sequential([keras.Input(shape=(n_entradas,))])
    for n in capas:
        modelo.add(keras.layers.Dense(n, activation="relu"))
    modelo.add(keras.layers.Dense(1))
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=tasa),
                   loss="mse", metrics=["mae"])
    return modelo

keras.backend.clear_session()        # reinicia el contador de nombres y libera memoria
keras.utils.set_random_seed(SEMILLA)
red_1 = construir_red(len(VARIABLES_1))
red_1.summary()

> 💬 **Verifiquemos el número de parámetros a mano.** Una capa densa con $n_{in}$ entradas y $n_{out}$ neuronas tiene $n_{in}\cdot n_{out} + n_{out}$ parámetros (pesos + sesgos):
> $14\cdot32+32 = 480$, $\;32\cdot16+16 = 528$, $\;16\cdot1+1 = 17$. Total: **1025**. Con 723 días de entrenamiento, ¡la red tiene más parámetros que ejemplos!



### 5.2 Entrenamiento

- `compile` ya fijó la pérdida (MSE) y el optimizador (Adam, $\eta = 10^{-3}$).
- `fit` ejecuta el ciclo de entrenamiento por lotes de 32 ejemplos.
- **Parada temprana (`EarlyStopping`)**: vigila la pérdida en *validación*; si no mejora en 30 épocas seguidas, detiene el entrenamiento y recupera los mejores pesos. Es la herramienta más simple contra el **sobreajuste** (memorizar el entrenamiento en lugar de aprender patrones generales).

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True)
reporte = keras.callbacks.LambdaCallback(
    on_epoch_end=lambda ep, logs: print(f"Época {ep+1:3d}  pérdida entren. = {logs['loss']:.4f}  pérdida valid. = {logs['val_loss']:.4f}")
    if (ep + 1) % 20 == 0 else None)

hist_1 = red_1.fit(X_ent1, y_ent1, validation_data=(X_val1, y_val1),
                   epochs=400, batch_size=32, verbose=0, callbacks=[parada, reporte])
print(f"\nEntrenamiento detenido tras {len(hist_1.history['loss'])} épocas.")

### 5.3 Curvas de aprendizaje

Comparar la pérdida de entrenamiento y la de validación es la forma básica de diagnosticar el entrenamiento:

- Ambas bajan juntas → la red está aprendiendo patrones generales.
- La de entrenamiento sigue bajando y la de validación sube → **sobreajuste**.

In [ ]:
def graficar_curvas(hist, titulo):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(hist.history["loss"], label="Entrenamiento")
    ax.plot(hist.history["val_loss"], label="Validación")
    ax.set_yscale("log"); ax.set_xlabel("Época"); ax.set_ylabel("MSE (escala estandarizada)")
    ax.set_title(titulo); ax.legend()

graficar_curvas(hist_1, "Red 1: curvas de aprendizaje")

In [ ]:
pred_red_1 = esc_y1.inverse_transform(red_1.predict(X_pru1, verbose=0)).ravel()
evaluar("Red 1 (objetivo directo)", y_prueba, pred_red_1)

### 5.4 Diagnóstico: ¿por qué la red no le gana a la regresión lineal?

¡Sorpresa! La red, con más de mil parámetros, pierde contra la regresión lineal. Una pista ya estaba en las curvas de aprendizaje: la pérdida de validación quedó muy por encima de la de entrenamiento. Recuerden la **Observación 1** y la palabra *acotado* del teorema de aproximación universal. Veamos el rango de la variable `nivel` en cada conjunto:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(entren["nivel"], bins=40, alpha=0.6, label="Entrenamiento (2021–2022)")
ax.hist(prueba["nivel"], bins=40, alpha=0.6, label="Prueba (2024)", color="C3")
ax.set_xlabel("Nivel de demanda: media de los 7 días previos (GWh/día)"); ax.legend()
ax.set_title("La prueba ocurre en una región que la red nunca vio");
fuera = (prueba["nivel"] > entren["nivel"].max()).mean()
print(f"Fracción de días de 2024 con nivel mayor que el máximo de entrenamiento: {100*fuera:.0f} %")

**Diagnóstico.** Por la tendencia creciente, en 2024 la demanda está casi siempre **por encima** de todo lo que la red vio al entrenar. Las redes neuronales interpolan muy bien dentro de la región de los datos, pero **extrapolan mal** fuera de ella: más allá del rango de entrenamiento, una red ReLU se comporta como una función lineal a trozos con pendientes que nadie ajustó para esa región. La regresión lineal, en cambio, extrapola su tendencia de forma "natural".

**Lección:** una red neuronal no sustituye el pensamiento sobre el problema. ¿Podemos reformular el objetivo para que la red trabaje en una región que sí conoce?

## 6. Segunda red: reformular el objetivo

### 6.1 Idea: modelar la demanda *relativa*

En lugar de predecir la demanda en GWh, predecimos qué tan por encima o por debajo estará del nivel reciente:

$$
r_t = \frac{y_t}{\text{nivel}_t}, \qquad \text{nivel}_t = \frac{1}{7}\sum_{k=1}^{7} y_{t-k}.
$$

- Si $r_t = 0{,}90$, ese día la demanda es 10 % menor que el promedio de la semana anterior (típico de un domingo o festivo).
- $r_t$ casi no tiene tendencia: su rango en 2024 es similar al de 2021–2022.
- Para recuperar el pronóstico en GWh basta con $\hat y_t = \hat r_t \cdot \text{nivel}_t$.

Expresamos también los rezagos en términos relativos: `r_lag1` $= y_{t-1}/\text{nivel}_t$ y `r_lag7` $= y_{t-7}/\text{nivel}_t$.

In [ ]:
for p in (entren, valid, prueba):
    p["r"] = p["demanda"] / p["nivel"]
    p["r_lag1"] = p["lag1"] / p["nivel"]
    p["r_lag7"] = p["lag7"] / p["nivel"]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(entren["r"], bins=40, alpha=0.6, label="Entrenamiento (2021–2022)")
ax.hist(prueba["r"], bins=40, alpha=0.6, label="Prueba (2024)", color="C3")
ax.set_xlabel("Demanda relativa r = demanda / nivel"); ax.legend()
ax.set_title("Ahora entrenamiento y prueba viven en la misma región");

In [ ]:
VARIABLES_2 = VARIABLES_CAL + ["r_lag1", "r_lag7"]
OBJETIVO_2 = "r"

esc_X2 = StandardScaler().fit(entren[VARIABLES_2])
esc_y2 = StandardScaler().fit(entren[[OBJETIVO_2]])
X_ent2, X_val2, X_pru2 = (esc_X2.transform(p[VARIABLES_2]) for p in (entren, valid, prueba))
y_ent2, y_val2 = (esc_y2.transform(p[[OBJETIVO_2]]) for p in (entren, valid))

def a_GWh(r_escalado, parte, escalador):
    # Deshace el escalado de r y multiplica por el nivel para volver a GWh/día
    return escalador.inverse_transform(r_escalado).ravel() * parte["nivel"].values

# Referencia justa: regresión lineal con la misma reformulación
lineal_2 = LinearRegression().fit(X_ent2, y_ent2)
evaluar("Regresión lineal (objetivo relativo)", y_prueba, a_GWh(lineal_2.predict(X_pru2), prueba, esc_y2))

In [ ]:
keras.utils.set_random_seed(SEMILLA)
red_2 = construir_red(len(VARIABLES_2))      # misma arquitectura que la red 1
hist_2 = red_2.fit(X_ent2, y_ent2, validation_data=(X_val2, y_val2),
                   epochs=400, batch_size=32, verbose=0, callbacks=[parada, reporte])
print(f"\nEntrenamiento detenido tras {len(hist_2.history['loss'])} épocas.")
graficar_curvas(hist_2, "Red 2: curvas de aprendizaje")

In [ ]:
pred_red_2 = a_GWh(red_2.predict(X_pru2, verbose=0), prueba, esc_y2)
evaluar("Red 2 (objetivo relativo)", y_prueba, pred_red_2)

### 6.2 Comparación final en 2024 (datos nunca vistos)

In [ ]:
tabla = pd.DataFrame(resultados).T.sort_values("MAPE (%)", ascending=False)
ax = tabla["MAPE (%)"].plot.barh(figsize=(9, 3.5), color=["C7"] * (len(tabla) - 1) + ["C2"])
ax.set_xlabel("MAPE en 2024 (%) — menor es mejor"); ax.set_title("Comparación de modelos")
for i, v in enumerate(tabla["MAPE (%)"]):
    ax.text(v + 0.03, i, f"{v:.2f} %", va="center")
tabla

### 6.3 Mirar los pronósticos, no solo las métricas

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(prueba.index, y_prueba, lw=1, label="Real", color="k")
axes[0].plot(prueba.index, pred_red_2, lw=1, label="Red 2", color="C2", alpha=0.85)
axes[0].set_title("Demanda real vs. pronóstico de la red 2 — 2024"); axes[0].set_ylabel("GWh/día"); axes[0].legend()

ventana = slice("2024-12-01", "2024-12-31")
axes[1].plot(prueba.loc[ventana].index, y_prueba[prueba.index.isin(prueba.loc[ventana].index)], "o-", color="k", label="Real")
axes[1].plot(prueba.loc[ventana].index, pred_red_2[prueba.index.isin(prueba.loc[ventana].index)], "s--", color="C2", label="Red 2")
for d in prueba.loc[ventana].index[prueba.loc[ventana, "festivo"] == 1]:
    axes[1].axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12), color="C1", alpha=0.2)
axes[1].set_title("Zoom: diciembre de 2024 (franjas naranjas = festivos)"); axes[1].set_ylabel("GWh/día"); axes[1].legend()
plt.tight_layout();

> 💬 **Para discutir:** la red falla sobre todo el 24 y el 31 de diciembre. No son festivos oficiales, así que la red los trata como días normales, pero en la práctica la actividad cae. ¿Qué variable añadirían para capturarlo? El modelo solo sabe lo que le contamos a través de las variables de entrada.

In [ ]:
errores = pd.DataFrame({
    "error_pct": 100 * np.abs(y_prueba - pred_red_2) / y_prueba,
    "tipo": np.where(prueba["festivo"] == 1, "Festivo",
             np.where(prueba["dia_semana"] == 6, "Domingo",
             np.where(prueba["dia_semana"] == 5, "Sábado", "Lun–Vie")))
})
print("Error porcentual medio de la red 2 por tipo de día (2024):")
errores.groupby("tipo")["error_pct"].agg(["mean", "count"]).round(2)

### 6.4 "No hay magia": reproducir una predicción a mano

Una red entrenada es solo una función con parámetros conocidos. Tomemos los pesos de la red 2 y calculemos la predicción de un día con NumPy, siguiendo las fórmulas de la sección 2.3.

In [ ]:
(W1, b1), (W2, b2), (W3, b3) = [capa.get_weights() for capa in red_2.layers]
x0 = X_pru2[0]                                   # primer día de 2024 (variables escaladas)

h1 = np.maximum(0, x0 @ W1 + b1)                 # capa oculta 1 (ReLU)
h2 = np.maximum(0, h1 @ W2 + b2)                 # capa oculta 2 (ReLU)
salida = h2 @ W3 + b3                            # salida lineal

print("A mano  :", salida.item())
print("Keras   :", red_2.predict(X_pru2[:1], verbose=0).item())
print(f"Neuronas activas: capa 1 = {(h1 > 0).sum()}/32, capa 2 = {(h2 > 0).sum()}/16")

## 7. Cierre

### Ideas finales

1. **Una red neuronal es una función paramétrica**: capas de transformaciones afines seguidas de activaciones no lineales. Lo comprobamos reproduciendo una predicción a mano.
2. **Se entrena minimizando una pérdida** con descenso de gradiente; Keras calcula los gradientes por retropropagación.
3. **La preparación de los datos es la mitad del trabajo**: codificación del calendario, rezagos, partición cronológica, escalado ajustado solo en entrenamiento y cuidado con la fuga de información.
4. **Siempre hay que comparar contra modelos de referencia.** Una red más compleja no garantiza un mejor resultado.
5. **Las redes interpolan bien y extrapolan mal.** Reformular el objetivo (demanda relativa en lugar de absoluta) puso a la red en una región conocida, y entonces sí aprovechó su capacidad para modelar interacciones no lineales (festivo × día de la semana, efecto de festivos en los rezagos).

### Limitaciones de lo que hicimos

- Pronóstico a **un** día: pronosticar una semana adelante es más difícil, porque los rezagos recientes no estarían disponibles.
- Una sola semilla aleatoria: los resultados de una red varían ligeramente entre ejecuciones y equipos. Un análisis serio promedia varias semillas.
- No incluimos variables exógenas como temperatura, actividad económica o el fenómeno de El Niño.

### Ejercicios propuestos

1. **Arquitectura:** prueben la red 2 con `capas=(8,)`, `capas=(64, 32)` y `capas=(64, 64, 32)`. ¿Más grande es siempre mejor? Usen el MAPE de **validación** para decidir, no el de prueba.
2. **Tasa de aprendizaje:** entrenen con `tasa=1e-2` y `tasa=1e-4`. ¿Cómo cambian las curvas de aprendizaje y el número de épocas?
3. **Semillas:** repitan el entrenamiento de la red 2 con 5 semillas distintas y reporten la media y la desviación estándar del MAPE.
4. **Precio de bolsa:** añadan el precio del día anterior (`precio.shift(1)`, quizá en logaritmo) como variable. ¿Mejora el pronóstico? Expliquen el resultado.
5. **Puentes festivos:** creen una variable que indique si el día siguiente o el anterior es festivo. ¿Mejora el error en los días cercanos a festivos?
6. **Reto:** adapten el notebook para pronosticar la demanda con 7 días de anticipación. ¿Qué variables dejan de ser válidas?